# Cosmic web v0 — mapping voids with the tidal-tensor (T-web) method

A minimal, runnable first step toward a 3D map of the cosmic web. We segment a density field into four classes — **void, sheet (wall), filament, knot (cluster)** — using the classical tidal-tensor classifier. No quantum, no GPU required; the whole thing runs in a few seconds in Colab.

**The idea.** Take a density field δ(x). Solve for the potential (∇²φ = δ), take its Hessian (the tidal/deformation tensor), and at each point find the three eigenvalues. Count how many exceed a threshold λ_th:

| eigenvalues > λ_th | collapsing axes | class |
|---|---|---|
| 0 | none → expanding everywhere | **void** |
| 1 | one | **sheet / wall** |
| 2 | two | **filament** |
| 3 | all three | **knot / cluster** |

This is exactly the labeling that modern ML void-finders (e.g. DeepVoid) feed to a 3D U-Net. We start on a *synthetic* field so it runs immediately, then there's a clearly marked section to swap in real QUIJOTE simulation data.

*Refs:* QUIJOTE simulations (`quijote-simulations.readthedocs.io`), Pylians (`github.com/franciscovillaescusa/Pylians3`), DeepVoid (arXiv:2504.21134).

## 0. Parameters — your one place to tweak

Small project, single knob-box. Lower `N` for faster iteration; raise it for finer maps. `lam_th` controls the void/filament balance (higher → more void volume).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- config ---
N        = 128      # grid cells per side (128 -> ~6s; try 64 while experimenting)
L        = 250.0    # box size in Mpc/h
sigma_g  = 1.2      # rms of the underlying Gaussian field (sets how 'clumpy')
R_smooth = 3.0      # smoothing scale in Mpc/h (the scale at which the web is defined)
lam_th   = 0.3      # eigenvalue threshold (0.2-0.4 typical; higher = more void)
seed     = 42

CLASS_NAMES  = ['void', 'sheet', 'filament', 'knot']
CLASS_COLORS = ['#08306b', '#6baed6', '#fd8d3c', '#cb181d']  # cold/empty -> hot/dense

## 1. A synthetic cosmic-web density field

We build a Gaussian random field with a CDM-like power spectrum (BBKS transfer function), then apply a lognormal transform so the density is positive with deep voids and dense peaks — a cheap but convincing stand-in for the real thing.

In [ ]:
def Pk_bbks(k, ns=0.96, Gamma=0.2):
    '''Crude CDM power spectrum: P(k) = k^ns * T(k)^2 (BBKS transfer function).'''
    k = np.asarray(k, dtype=float)
    q = np.where(k > 0, k / Gamma, 1e-10)
    T = (np.log(1 + 2.34*q) / (2.34*q)) * \
        (1 + 3.89*q + (16.1*q)**2 + (5.46*q)**3 + (6.71*q)**4)**(-0.25)
    return k**ns * T**2

def make_lognormal_field(N, L, sigma_g=1.0, seed=42):
    '''Return a lognormal density-contrast field delta(x) on an N^3 grid.'''
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi          # angular wavenumbers (h/Mpc)
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    kmag = np.sqrt(KX**2 + KY**2 + KZ**2); kmag[0,0,0] = 1.0
    rng = np.random.default_rng(seed)
    white_k = np.fft.fftn(rng.standard_normal((N, N, N)))
    Pk = Pk_bbks(kmag); Pk[0,0,0] = 0.0
    g = np.fft.ifftn(white_k * np.sqrt(Pk)).real      # Gaussian field with P(k)
    g -= g.mean(); g *= sigma_g / g.std()
    sg = g.std()
    return np.exp(g - 0.5*sg**2) - 1.0                # lognormal -> mean(1+delta)=1

delta = make_lognormal_field(N, L, sigma_g=sigma_g, seed=seed)
print(f"delta: min={delta.min():.2f}  max={delta.max():.1f}  mean={delta.mean():.3f}  std={delta.std():.2f}")

## 2. The T-web classifier

Three small functions: smooth the field, build the tidal tensor by FFT, then count eigenvalues above threshold. The tensor is normalised so its trace equals δ (the eigenvalues sum to the density contrast) — a handy sanity check.

In [ ]:
def smooth_gauss(delta, L, R):
    N = delta.shape[0]
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    k2 = KX**2 + KY**2 + KZ**2
    return np.fft.ifftn(np.fft.fftn(delta) * np.exp(-0.5 * k2 * R**2)).real

def tidal_tensor(delta, L):
    '''Deformation tensor T_ij = d^2 phi / dx_i dx_j, with trace(T) = delta.'''
    N = delta.shape[0]
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    Kv = [KX, KY, KZ]
    k2 = KX**2 + KY**2 + KZ**2; k2[0,0,0] = 1.0
    dk = np.fft.fftn(delta)
    T = np.zeros((N, N, N, 3, 3))
    for i in range(3):
        for j in range(i, 3):
            Tij = np.fft.ifftn((Kv[i]*Kv[j]/k2) * dk).real
            T[..., i, j] = Tij
            if i != j: T[..., j, i] = Tij
    return T

def classify_web(T, lam_th=0.0):
    eigs = np.linalg.eigvalsh(T)                 # ascending eigenvalues, shape (N,N,N,3)
    n_above = np.sum(eigs > lam_th, axis=-1)     # 0..3  ==  class index
    return n_above.astype(np.int8), eigs

delta_s = smooth_gauss(delta, L, R_smooth)
T       = tidal_tensor(delta_s, L)
labels, eigs = classify_web(T, lam_th=lam_th)

print(f"trace check (max |sum(eig) - delta_s|): {np.abs(eigs.sum(-1) - delta_s).max():.1e}\n")
for c in range(4):
    print(f"  {CLASS_NAMES[c]:9s}: {100*(labels==c).mean():5.1f}% of volume")

## 3. Look at the map

A central 2D slice: the density on the left, the four-class web on the right. Voids should fill most of the area, with filaments threading between rare knots.

In [ ]:
z = N // 2
dens_slice  = np.log10(np.clip(1 + delta, 1e-3, None))[:, :, z]
label_slice = labels[:, :, z]

cmap = ListedColormap(CLASS_COLORS)
norm = BoundaryNorm([-.5, .5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
im0 = ax[0].imshow(dens_slice.T, origin='lower', cmap='inferno',
                   extent=[0, L, 0, L], vmin=np.percentile(dens_slice, 2))
ax[0].set_title(r'density   log$_{10}$(1+$\delta$)')
plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

im1 = ax[1].imshow(label_slice.T, origin='lower', cmap=cmap, norm=norm, extent=[0, L, 0, L])
ax[1].set_title('cosmic-web class')
cb = plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04, ticks=[0, 1, 2, 3])
cb.ax.set_yticklabels(CLASS_NAMES)

for a in ax:
    a.set_xlabel('Mpc/h'); a.set_ylabel('Mpc/h')
plt.tight_layout(); plt.show()

## 4. Upgrade: a realistic field you generate locally — no download

QUIJOTE's data sits behind Globus/Binder, and you want to stay local — so here's the better mock: the **Zel'dovich approximation**, the standard first step of any N-body simulation. Take the linear density field, compute the gravitational displacement it implies, and slide a grid of particles along it. One step of real gravity turns the round lognormal blobs into the *anisotropic, connected* filaments and sheets that define the cosmic web. Still a simulation rather than real data — but a genuine step up, in a few seconds of pure NumPy.

`rms_disp` (typical particle displacement, Mpc/h) is the knob: larger = more evolved and stringier, until shell-crossing starts to smear it (~10-12 is a sensible ceiling). Run this cell, then re-run the Step-3 plot to see the map.

In [ ]:
# Zel'dovich: displace a particle grid by the gravitational displacement, then
# deposit with cloud-in-cell. Reuses Pk_bbks / smooth_gauss / tidal_tensor / classify_web.

def cic_deposit(pos, N, L):
    g = np.zeros((N, N, N))
    s = (pos % L) / (L / N)                      # positions in cell units, [0, N)
    i = np.floor(s).astype(int) % N
    f = s - np.floor(s)
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                wx = f[:, 0] if dx else 1 - f[:, 0]
                wy = f[:, 1] if dy else 1 - f[:, 1]
                wz = f[:, 2] if dz else 1 - f[:, 2]
                ii = (i[:, 0] + dx) % N; jj = (i[:, 1] + dy) % N; kk = (i[:, 2] + dz) % N
                np.add.at(g, (ii, jj, kk), wx * wy * wz)
    return g

def zeldovich_delta(N, L, rms_disp=8.0, seed=42):
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    Kv = [KX, KY, KZ]; k2 = KX**2 + KY**2 + KZ**2; k2[0, 0, 0] = 1.0
    rng = np.random.default_rng(seed)
    dlin_k = np.fft.fftn(rng.standard_normal((N, N, N))) * np.sqrt(Pk_bbks(np.sqrt(k2)))
    dlin_k[0, 0, 0] = 0.0
    q = np.arange(N) * (L/N); QX, QY, QZ = np.meshgrid(q, q, q, indexing='ij'); Q = [QX, QY, QZ]
    disp = np.array([np.fft.ifftn(1j * Kv[i] * dlin_k / k2).real for i in range(3)])  # displacement Psi
    disp *= rms_disp / np.sqrt((disp**2).sum(0)).std()        # scale to target rms (Mpc/h)
    pos = np.stack([(Q[i] + disp[i]).ravel() for i in range(3)], axis=1)
    rho = cic_deposit(pos, N, L)
    return rho / rho.mean() - 1.0

rms_disp = 8.0                                    # the knob: bigger = more evolved / stringier
delta_za = zeldovich_delta(N, L, rms_disp=rms_disp, seed=seed)
delta_s  = smooth_gauss(delta_za, L, R_smooth)
labels, eigs = classify_web(tidal_tensor(delta_s, L), lam_th=lam_th)
delta = delta_za                                  # so the Step-3 plot cell shows THIS field

print(f"Zeldovich delta: min={delta.min():.2f}  max={delta.max():.0f}  std={delta.std():.2f}")
for c in range(4):
    print(f"  {CLASS_NAMES[c]:9s}: {100*(labels==c).mean():5.1f}% of volume")
print("\n-> scroll up and re-run the Step-3 plot cell to see this map")

## 5. (Optional) real QUIJOTE data — no Pylians needed

The Pylians install failed because it compiles C/Cython and wants Cython ≥ 3.0 plus a C/OpenMP toolchain — painful on Windows + Python 3.13. We don't actually need it: QUIJOTE files are HDF5 / `.npy`, readable with tools that install from wheels (no compiler).

Three routes, easiest first:

- **A — precomputed density field (recommended).** Download one `df_m_256_CIC_z=0.npy` from QUIJOTE's *Density fields* product. `np.load` reads it with nothing but NumPy — the CIC gridding is already done. Run **cell A**.
- **B — raw snapshot.** If you grabbed a snapshot instead, read `PartType1/Coordinates` with `h5py` + `hdf5plugin` and grid it with NumPy. Run the install cell, then **cell B**.
- **C — zero install, zero download.** Skip the laptop entirely: QUIJOTE's *Binder* (linked from their Data-access docs) runs in-browser with the data mounted and Pylians preinstalled.

QUIJOTE boxes are 1000 Mpc/h; at 256³ a voxel is ~3.9 Mpc/h, so smooth at a few voxels (R≈8), not the R=3 we used for the mock. After running a cell below it sets `delta`, `labels`, `L` to the real data — scroll up and re-run the Step-3 plot cell to see the real map.

In [ ]:
# Only needed for route B (raw snapshot). Route A (density field) needs only numpy.
# Both install from wheels on Windows -- no compiler required.
!pip install h5py hdf5plugin

In [ ]:
import os, numpy as np

# Route A: precomputed CIC density field (df_m_256_CIC_z=0.npy from QUIJOTE 'Density fields')
DF_PATH  = r"C:\path\to\df_m_256_CIC_z=0.npy"   # <-- edit me
BoxSize  = 1000.0     # QUIJOTE box, Mpc/h
R_real   = 8.0        # smoothing ~2 voxels at 256^3; tune
lam_real = 0.2        # tune

def to_delta(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.min() < -0.5 and abs(arr.mean()) < 0.1:
        return arr                        # already an overdensity field
    return arr / arr.mean() - 1.0         # particle counts -> overdensity

if not os.path.exists(DF_PATH):
    print("No density field at:", DF_PATH)
    print("Grab df_m_256_CIC_z=0.npy from QUIJOTE 'Density fields' (Globus or Binder),")
    print("set DF_PATH above, then re-run this cell.")
else:
    delta_real = to_delta(np.load(DF_PATH))
    print(f"loaded {delta_real.shape}  delta range [{delta_real.min():.2f}, {delta_real.max():.1f}]")
    delta_s_r   = smooth_gauss(delta_real, BoxSize, R_real)
    T_r         = tidal_tensor(delta_s_r, BoxSize)
    labels_r, _ = classify_web(T_r, lam_th=lam_real)
    for c in range(4):
        print(f"  {CLASS_NAMES[c]:9s}: {100*(labels_r==c).mean():5.1f}%")
    delta, labels, L = delta_real, labels_r, BoxSize   # -> re-run the Step-3 plot cell

In [ ]:
import os, glob, numpy as np

# Route B: raw QUIJOTE snapshot (HDF5, Blosc-compressed -> needs hdf5plugin)
SNAPDIR = r"C:\path\to\snapdir_004"   # <-- edit me (folder holding snap_004.*.hdf5)
Ngrid   = 256

files = sorted(glob.glob(os.path.join(SNAPDIR, "*.hdf5")))
if not files:
    print("No .hdf5 files in:", SNAPDIR)
    print("If your files are raw Gadget-II binary, route A (density field) is much simpler.")
else:
    import h5py, hdf5plugin            # hdf5plugin registers the Blosc decompressor
    pos, BoxSize = [], None
    for fn in files:
        with h5py.File(fn, "r") as f:
            if BoxSize is None:
                BoxSize = float(f["Header"].attrs["BoxSize"]) / 1e3   # kpc/h -> Mpc/h
            pos.append(f["PartType1/Coordinates"][:] / 1e3)           # CDM positions, Mpc/h
    pos = np.concatenate(pos)
    print(f"{len(pos):,} particles, box {BoxSize:.0f} Mpc/h")
    counts, _   = np.histogramdd(pos, bins=Ngrid, range=[[0, BoxSize]]*3)   # NGP gridding
    delta_real  = counts / counts.mean() - 1.0
    delta_s_r   = smooth_gauss(delta_real, BoxSize, 8.0)
    T_r         = tidal_tensor(delta_s_r, BoxSize)
    labels_r, _ = classify_web(T_r, lam_th=0.2)
    for c in range(4):
        print(f"  {CLASS_NAMES[c]:9s}: {100*(labels_r==c).mean():5.1f}%")
    delta, labels, L = delta_real, labels_r, BoxSize   # -> re-run the Step-3 plot cell

## Next steps

- **Tune & explore.** Slide `lam_th`, `R_smooth`, `sigma_g`; watch the volume fractions and morphology shift. Real N-body data gives crisper, more anisotropic filaments than the lognormal mock.
- **The ML step (the actual research question).** Train a 3D U-Net to predict this label cube from a *sparser* input — a halo-count field or subsampled density — and score it with F1 / IoU / Matthews correlation coefficient against the ground truth. That's the DeepVoid recipe on your own data.
- **Cross-check the voids — no extra software.** QUIJOTE ships void catalogs as HDF5; read them with h5py (`pos`, `radius`, `VSF`) and overlay on the void class here.
- **Go observational.** Apply the trained model to a DESI DR1 region (`data.desi.lbl.gov/public/dr1`), or run VIDE (on Binder/Colab, where it installs cleanly) on the DESI large-scale-structure catalog for a real void map.